# 00 — Setup e Inicialización del Clúster

Este notebook verifica la conexión con Hadoop HDFS y el clúster Spark,
carga los datos en memoria y realiza una exploración inicial del dataset.

**Infraestructura:**
- Hadoop HDFS: `hdfs://namenode:8020`
- Spark Master: `spark://spark-master:7077`
- Dataset: Olist E-commerce (sintético, ~100k órdenes)

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

HDFS         = os.environ.get('HADOOP_NAMENODE', 'hdfs://namenode:8020')
SPARK_MASTER = os.environ.get('SPARK_MASTER', 'local[*]')
DATA_PATH    = f'{HDFS}/data/olist'
OUTPUT_PATH  = '/home/jovyan/work/outputs'

print(f'HDFS:          {HDFS}')
print(f'Spark Master:  {SPARK_MASTER}')
print(f'Data path:     {DATA_PATH}')

HDFS:          hdfs://namenode:8020
Spark Master:  local[*]
Data path:     hdfs://namenode:8020/data/olist


In [2]:
spark = SparkSession.builder \
    .appName('00_Setup_Olist') \
    .master(SPARK_MASTER) \
    .config('spark.hadoop.fs.defaultFS', HDFS) \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.driver.memory', '1g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version:  {spark.version}')
print(f'Spark UI:       http://localhost:4040')

26/04/17 17:34:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version:  3.0.3
Spark UI:       http://localhost:4040


## Verificación de archivos en HDFS

In [3]:
tables = ['orders', 'order_items', 'order_payments', 'order_reviews',
          'customers', 'products', 'sellers']

print(f'{"Tabla":<25} {"Filas":>10}')
print('-' * 37)
dfs = {}
for t in tables:
    df = spark.read.csv(f'{DATA_PATH}/{t}.csv', header=True, inferSchema=True)
    dfs[t] = df
    print(f'{t:<25} {df.count():>10,}')

Tabla                          Filas
-------------------------------------
orders                       100,000
order_items                  131,913
order_payments               100,000
order_reviews                 62,802
customers                     95,000
products                      32,000
sellers                        3,000


## Schema de tablas principales

In [4]:
print('=== ORDERS ===')
dfs['orders'].printSchema()

print('=== ORDER_ITEMS ===')
dfs['order_items'].printSchema()

print('=== ORDER_PAYMENTS ===')
dfs['order_payments'].printSchema()

=== ORDERS ===
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

=== ORDER_ITEMS ===
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

=== ORDER_PAYMENTS ===
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



## Estadísticas generales del dataset

In [5]:
orders   = dfs['orders']
payments = dfs['order_payments']

total_revenue = payments.agg(F.sum('payment_value').alias('total')).collect()[0]['total']
avg_ticket    = payments.agg(F.avg('payment_value').alias('avg')).collect()[0]['avg']
status_counts = orders.groupBy('order_status').count().orderBy(F.desc('count'))

print(f'Revenue total:    $ {total_revenue:>15,.2f}')
print(f'Ticket promedio:  $ {avg_ticket:>15,.2f}')
print()
print('Distribución por estado de orden:')
status_counts.show()

Revenue total:    $   24,089,542.72
Ticket promedio:  $          240.90

Distribución por estado de orden:
+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|62802|
|    canceled|12452|
|     shipped|12423|
|  processing|12323|
+------------+-----+



In [6]:
date_range = orders.agg(
    F.min('order_purchase_timestamp').alias('primera_orden'),
    F.max('order_purchase_timestamp').alias('ultima_orden')
).collect()[0]

print(f"Período: {date_range['primera_orden']}  →  {date_range['ultima_orden']}")

Período: 2017-01-01 00:04:53  →  2018-12-30 23:51:55


## Verificación de valores nulos

In [7]:
for name, df in dfs.items():
    nulls = {c: df.filter(F.col(c).isNull()).count() for c in df.columns}
    has_nulls = {k: v for k, v in nulls.items() if v > 0}
    if has_nulls:
        print(f'{name}: {has_nulls}')
    else:
        print(f'{name}: sin nulos ✓')

orders: sin nulos ✓
order_items: sin nulos ✓
order_payments: sin nulos ✓
order_reviews: sin nulos ✓
customers: sin nulos ✓
products: sin nulos ✓
sellers: sin nulos ✓


In [8]:
spark.stop()
print('Sesión Spark cerrada. Setup completo.')

Sesión Spark cerrada. Setup completo.
